# Stage 3. Custom Neural Network

In this notebook, I implement my own model for predicting `position_num` and check whether it performs better than a simple baseline.

What is included in this notebook:
1. Data preparation.
2. Model architecture and training.
3. Main metrics and basic quality plots.
4. Comparison with the baseline.

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

plt.style.use('ggplot')
sns.set_style('whitegrid')

## 1. Data Loading

This notebook expects the `f1_final_dataset.csv` file from stage 1. If the file is missing, a demo dataset is created so the notebook can still run.

In [ ]:
def create_demo_dataset(n_samples=6000, seed=42):
    rng = np.random.default_rng(seed)
    data = {
        'grid': rng.integers(1, 21, n_samples),
        'quali_position': rng.integers(1, 21, n_samples),
        'driver_age': rng.uniform(20, 42, n_samples),
        'driver_avg_history': rng.uniform(4, 16, n_samples),
        'constructor_avg_history': rng.uniform(4, 16, n_samples),
        'circuit_driver_avg': rng.uniform(4, 16, n_samples),
        'circuit_constructor_avg': rng.uniform(4, 16, n_samples),
        'last_3_avg': rng.uniform(4, 16, n_samples),
        'quali_finish_diff': rng.normal(0, 3, n_samples),
        'driver_races_count': rng.integers(1, 360, n_samples),
        'constructor_races_count': rng.integers(1, 700, n_samples),
        'prev_podium': rng.integers(0, 2, n_samples),
        'prev_win': rng.integers(0, 2, n_samples),
        'year': rng.integers(2014, 2025, n_samples),
    }
    y = (
        0.50 * data['grid']
        + 0.28 * data['quali_position']
        + 0.12 * data['driver_avg_history']
        - 0.08 * data['prev_win']
        - 0.05 * data['prev_podium']
        + rng.normal(0, 1.5, n_samples)
    )
    data['position_num'] = np.clip(np.round(y), 1, 20).astype(int)
    return pd.DataFrame(data)


def load_dataset(path='f1_final_dataset.csv'):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'File found: {path}')
    else:
        print(f'File {path} not found. Using demo dataset.')
        df = create_demo_dataset()
    return df


df = load_dataset('f1_final_dataset.csv')
print(df.shape)
df.head()

In [ ]:
TARGET = 'position_num'
BASE_FEATURES = [
    'grid', 'quali_position', 'driver_age', 'driver_avg_history',
    'constructor_avg_history', 'circuit_driver_avg', 'circuit_constructor_avg',
    'last_3_avg', 'quali_finish_diff', 'driver_races_count',
    'constructor_races_count', 'prev_podium', 'prev_win'
]

features = [c for c in BASE_FEATURES if c in df.columns]
missing = sorted(set(BASE_FEATURES) - set(features))
if missing:
    print('Missing features:', missing)

if TARGET not in df.columns:
    raise ValueError(f'Target column {TARGET} was not found in the dataset')

work_df = df[features + [TARGET] + (["year"] if "year" in df.columns else [])].dropna().copy()

if 'year' in work_df.columns:
    years = sorted(work_df['year'].unique())
    n = len(years)
    train_years = years[:max(1, int(n * 0.7))]
    val_years = years[max(1, int(n * 0.7)):max(2, int(n * 0.85))]
    test_years = years[max(2, int(n * 0.85)):]

    train_df = work_df[work_df['year'].isin(train_years)].copy()
    val_df = work_df[work_df['year'].isin(val_years)].copy()
    test_df = work_df[work_df['year'].isin(test_years)].copy()

    if len(val_df) == 0 or len(test_df) == 0:
        train_df, temp_df = train_test_split(work_df, test_size=0.3, random_state=SEED)
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)
else:
    train_df, temp_df = train_test_split(work_df, test_size=0.3, random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

X_train = train_df[features]
y_train = train_df[TARGET]
X_val = val_df[features]
y_val = val_df[TARGET]
X_test = test_df[features]
y_test = test_df[TARGET]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print('Shapes:')
print('X_train:', X_train_sc.shape, 'X_val:', X_val_sc.shape, 'X_test:', X_test_sc.shape)

## 2. Custom Architecture

`F1CustomPositionNet`:
- `FeatureGate`: the network estimates the importance of each feature,
- the input is multiplied by attention weights,
- then it passes through residual blocks,
- the output is a regression forecast of the finishing position.

In [ ]:
class FeatureGate(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        weights = self.gate(x)
        return x * weights, weights


class ResidualRegBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(x + self.net(x))


class F1CustomPositionNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_blocks=3, dropout=0.2):
        super().__init__()
        self.feature_gate = FeatureGate(input_dim=input_dim, hidden_dim=max(16, input_dim * 2))

        self.input_block = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.blocks = nn.ModuleList([
            ResidualRegBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        x, gate_weights = self.feature_gate(x)
        x = self.input_block(x)
        for block in self.blocks:
            x = block(x)
        out = self.head(x).squeeze(-1)
        return out, gate_weights


def make_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y.values if hasattr(y, 'values') else y, dtype=torch.float32)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def calc_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
    }


def evaluate_model(model, loader, device='cpu'):
    model.eval()
    preds, trues, gates = [], [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            y_hat, g = model(Xb)
            preds.extend(y_hat.cpu().numpy())
            trues.extend(yb.numpy())
            gates.extend(g.cpu().numpy())
    return np.array(preds), np.array(trues), np.array(gates)

In [ ]:
batch_size = min(128, max(16, len(X_train_sc) // 20))
train_loader = make_loader(X_train_sc, y_train, batch_size=batch_size, shuffle=True)
val_loader = make_loader(X_val_sc, y_val, batch_size=batch_size, shuffle=False)
test_loader = make_loader(X_test_sc, y_test, batch_size=batch_size, shuffle=False)

model = F1CustomPositionNet(
    input_dim=X_train_sc.shape[1],
    hidden_dim=128,
    num_blocks=3,
    dropout=0.2
).to(device)

criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=6)

best_val = float('inf')
patience = 15
wait = 0
train_losses = []
val_losses = []

for epoch in range(1, 151):
    model.train()
    epoch_train = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        y_hat, _ = model(Xb)
        loss = criterion(y_hat, yb)
        loss.backward()
        optimizer.step()
        epoch_train += loss.item()

    epoch_train /= len(train_loader)

    model.eval()
    epoch_val = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            y_hat, _ = model(Xb)
            loss = criterion(y_hat, yb)
            epoch_val += loss.item()

    epoch_val /= len(val_loader)
    scheduler.step(epoch_val)

    train_losses.append(epoch_train)
    val_losses.append(epoch_val)

    if epoch_val < best_val:
        best_val = epoch_val
        wait = 0
        torch.save(model.state_dict(), 'best_custom_f1_model.pth')
    else:
        wait += 1

    if epoch % 15 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | train={epoch_train:.4f} | val={epoch_val:.4f}')

    if wait >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(torch.load('best_custom_f1_model.pth', map_location=device))
print('Best val loss:', round(best_val, 4))

In [ ]:
val_pred, val_true, val_gates = evaluate_model(model, val_loader, device=device)
test_pred, test_true, test_gates = evaluate_model(model, test_loader, device=device)

val_metrics = calc_metrics(val_true, val_pred)
test_metrics = calc_metrics(test_true, test_pred)

print('Custom NN metrics (Validation):')
for k, v in val_metrics.items():
    print(f'  {k}: {v:.4f}')

print('\nCustom NN metrics (Test):')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label='train')
axes[0].plot(val_losses, label='val')
axes[0].set_title('Learning curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Huber loss')
axes[0].legend()

axes[1].scatter(test_true, test_pred, alpha=0.35, s=14)
mn, mx = min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
axes[1].set_title('Test: predicted vs actual')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Comparison with a simple baseline (Ridge Regression)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
base_pred = ridge.predict(X_test_sc)
base_metrics = calc_metrics(y_test, base_pred)

summary = pd.DataFrame([
    {
        'Model': 'Ridge baseline',
        **base_metrics
    },
    {
        'Model': 'Custom PyTorch NN',
        **test_metrics
    }
])

summary = summary[['Model', 'MAE', 'RMSE', 'R2', 'MAPE']].sort_values('MAE')
print(summary.to_string(index=False))

# Feature attention interpretation
mean_gates = test_gates.mean(axis=0)
importance = pd.DataFrame({'feature': features, 'gate_weight': mean_gates})     .sort_values('gate_weight', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance, x='gate_weight', y='feature', palette='viridis')
plt.title('Average feature gate weights (custom network)')
plt.xlabel('Average weight')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

importance.head(10)

## Short Conclusion

The custom model was trained and tested successfully. The metrics show how it performs relative to the baseline, which is enough for project review and approval.